In [ ]:
!pip install -q sentence-transformers faiss-gpu-cu12 networkx pandas numpy scikit-learn tqdm

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("hf_token")

In [ ]:
from huggingface_hub import login
login(token=secret_value)

In [ ]:
import json
import pickle
import numpy as np
import faiss
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def load_graph_json(graph_file):
    print(f"Loading graph from {graph_file}...")
    with open(graph_file, 'r', encoding='utf-8') as f:
        graph_data = json.load(f)
    return graph_data

def generate_embeddings(graph_data, model_name="BAAI/bge-m3", batch_size=32):
    print(f"\nLoading model: {model_name}...")
    model = SentenceTransformer(model_name, device=device)
    
    if device == 'cuda':
        model.half()
    
    paragraph_nodes = [
        node for node in graph_data['nodes'] 
        if node.get('node_type') == 'PARAGRAPH'
    ]
    
    print(f"Found {len(paragraph_nodes)} paragraph nodes")
    
    texts_to_embed = []
    paragraph_info = {}
    paragraph_ids = []
    
    print("\nPreparing texts for embedding...")
    for node in tqdm(paragraph_nodes):
        node_id = node['id']
        section_title = node.get('section_title', '')
        label = node.get('label', '')
        content = node.get('content', '')
        
        text = f"{section_title}\n{label}\n{content}"
        
        texts_to_embed.append(text)
        paragraph_info[node_id] = {
            'section_title': section_title,
            'label': label,
            'content': content
        }
        paragraph_ids.append(node_id)
    
    print("\nGenerating embeddings...")
    embeddings = model.encode(
        texts_to_embed,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    embeddings = embeddings.astype(np.float32)
    faiss.normalize_L2(embeddings)
    
    print(f"Embeddings shape: {embeddings.shape}")
    
    print("Building FAISS index...")
    d = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(d)
    faiss_index.add(embeddings)
    print(f"FAISS index built with {faiss_index.ntotal} vectors")
    
    embeddings_dict = {}
    for i, paragraph_id in enumerate(paragraph_ids):
        embeddings_dict[paragraph_id] = {
            'embedding': embeddings[i].tolist(),
            **paragraph_info[paragraph_id]
        }
    
    return embeddings_dict, faiss_index, paragraph_ids, embeddings

def save_embeddings(embeddings_dict, faiss_index, paragraph_ids, embeddings_array, output_pickle, output_faiss):
    output_pickle.parent.mkdir(parents=True, exist_ok=True)
    output_faiss.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"\nSaving embeddings to pickle: {output_pickle}")
    pickle_data = {
        'embeddings_dict': embeddings_dict,
        'paragraph_ids': paragraph_ids
    }
    with open(output_pickle, 'wb') as f:
        pickle.dump(pickle_data, f)
    print(f"Saved pickle file")
    
    print(f"Saving FAISS index to: {output_faiss}")
    faiss.write_index(faiss_index, str(output_faiss))
    print(f"Saved FAISS index")

def print_stats(embeddings_dict):
    print("\n" + "="*60)
    print("EMBEDDING STATISTICS")
    print("="*60)
    print(f"Total paragraphs embedded: {len(embeddings_dict)}")
    
    first_embedding = list(embeddings_dict.values())[0]['embedding']
    print(f"Embedding dimension: {len(first_embedding)}")
    
    print("\nSample paragraph (first one):")
    sample_id = list(embeddings_dict.keys())[0]
    sample = embeddings_dict[sample_id]
    print(f"  Paragraph ID: {sample_id}")
    print(f"  Section Title: {sample['section_title']}")
    print(f"  Label: {sample['label']}")
    print(f"  Content: {sample['content'][:100]}...")
    print(f"  Embedding (first 5 dims): {sample['embedding'][:5]}")
    print("="*60 + "\n")

if __name__ == "__main__":
    try:
        graph_json = Path("/kaggle/input/datasets/minhvb10/zalo-dataset/legal_kg.json")
        output_pickle = Path("/kaggle/working/paragraph_embeddings.pkl")
        output_faiss = Path("/kaggle/working/paragraph_embeddings.faiss")
        
        print(f"Using input: {graph_json}")
        print(f"Output pickle: {output_pickle}")
        print(f"Output faiss: {output_faiss}")
        
        graph_data = load_graph_json(str(graph_json))
        
        embeddings_dict, faiss_index, paragraph_ids, embeddings_array = generate_embeddings(
            graph_data,
            model_name="BAAI/bge-m3",
            batch_size=32
        )
        
        save_embeddings(
            embeddings_dict,
            faiss_index,
            paragraph_ids,
            embeddings_array,
            output_pickle,
            output_faiss
        )
        
        print_stats(embeddings_dict)
        
        print("Embedding complete!")
        print(f"\nOutput files saved to: /kaggle/working/")
            
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()